In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day02" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w7" / "day02"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"
FORMATS = SAMPLES / "_formats"

프로젝트 루트  : c:\hw\hw\python_basic\hanwha-agent


In [2]:
from sqlalchemy import func, inspect, select
from app.db.session import get_engine, session_scope
from app.models.document import Chunk
from app.models.eval import EvalRun, EvalWeakType
SKIP = ('.git', '.pytest_cache', '.venv', '__pycache__', 'sandbox')
files = [p for p in ROOT.rglob('*') if p.is_file() and (not any((s in p.parts for s in SKIP)))]
migrations = sorted((ROOT / 'backend/app/db/migrations/versions').glob('*.py'))
table_names = inspect(get_engine()).get_table_names()
print(f'프로젝트 파일 : {len(files)}개   (.git · .pytest_cache · .venv · __pycache__ · sandbox 제외)')
print(f'chunks 컬럼   : {len(Chunk.__table__.columns)}        ← 어제와 같습니다. 오늘도 칸을 안 늘립니다')
print(f'DB 표         : {len(table_names)}       ← 어제 저녁에 9 에서 11 이 됐습니다')
print(f'마이그레이션  : {len(migrations)}건      ← 오늘은 0건이 늘 예정입니다')
if 'eval_runs' not in table_names:
    print('eval_runs        : 표가 없습니다 — 아래 복원 셀이 만듭니다')
else:
    with session_scope() as s:
        runs = s.scalar(select(func.count()).select_from(EvalRun))
        weak = s.scalar(select(func.count()).select_from(EvalWeakType))
    print(f'eval_runs        : {runs}행   ← 어제 「기본 검색」 한 줄')
    print(f'eval_weak_types  : {weak}행   ← 어제 저녁의 오답 유형 넷')


KeyboardInterrupt: 

In [ ]:
from sqlalchemy import select
from app.db.session import session_scope
from app.models.eval import EvalWeakType

with session_scope() as s:
    weak_rows = [(w.ord, w.label, w.count, w.accuracy, w.diagnosis) for w in s.scalars(select(EvalWeakType).order_by(EvalWeakType.ord))]
    
if not weak_rows:
    print('어제 남긴 오답 유형이 없습니다 — 지난시간에 테스트를 돌리지 않았거나 DB 를 새로 만든 것입니다.')
for (ord_, label, cnt, acc, diag) in weak_rows:
    print(f'[{ord_ + 1}] {label:<18}{cnt:>3}건   정답률 {acc:>5}   ← {diag}')

In [ ]:
import json

REPORTS = ROOT / 'evals' / 'reports'
saved = sorted(REPORTS.glob('eval_*.json'))
wrong = []

if not saved:
    print('어제 리포트가 없습니다 — 터미널에서 run_evals 명령을 --no-save 없이 한 번 더 치면 파일이 떨어집니다.')
else:
    latest = saved[-1]
    report = json.loads(latest.read_text(encoding='utf-8'))
    wrong = [r for r in report['runs'][0]['results'] if r['ok'] is False]
    print(f'어제 리포트 : {latest.relative_to(ROOT).as_posix()}')
    print(f'틀린 문항 {len(wrong)}개 중 셋만 봅니다')
    for r in wrong[:3]:
        print(f"  {r['id']}  {r['type']:<12} ← {r['why']}")


**질의 재작성과 Multi-Query 검색**

In [ ]:
from sqlalchemy import func, select
from app.db.session import session_scope
from app.models.document import Chunk
WORDS = ['출장비', '여비', '숙박', '1박당', '1박 당']
with session_scope() as s:
    for word in WORDS:
        n = s.scalar(select(func.count()).select_from(Chunk).where(Chunk.text.like(f'%{word}%')))
        print(f"LIKE '%{word}%'".ljust(16), '→', f'{n}건')

**질문을 고칠 수 없으면 질문을 늘리자.**